# 动手练习：社团结构与社区发现

来自《生物医学网络科学》（Biomedical Network Science）。

每个任务先运行“准备”格，再在“你的代码”格中完成 TODO，最后运行“核对”格：打印 ✓ 表示通过，否则会给出提示。

## 任务 1：亲手计算模块度

按社团求和的模块度公式：
$$Q = \sum_c\left[\frac{l_c}{m} - \left(\frac{d_c}{2m}\right)^2\right],$$
$l_c$ 为社团 $c$ 内部的边数，$d_c$ 为社团 $c$ 内所有节点的度之和，$m$ 为**全网**总边数。

补全函数 `my_modularity`，计算：

- `q_tri`：“两个三角形 + 一条桥”按两个三角形划分时的 $Q$（专题页手算约 0.357）；
- `q_club`：空手道俱乐部按真实分裂的两派划分时的 $Q$；
- `q_one`：空手道网络所有节点放在一个社团时的 $Q$（应为 0）。

并用 `nx.community.modularity` 核对。

> **提示：** `l_c = G.subgraph(c).number_of_edges()`；`d_c = sum(d for _, d in G.degree(c))`。

In [ ]:
# 准备（直接运行）
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

K = nx.karate_club_graph()
G = nx.Graph(K.edges())          # 去掉边权，按无权网络计算
club = [{n for n in K if K.nodes[n]["club"] == "Mr. Hi"},
        {n for n in K if K.nodes[n]["club"] != "Mr. Hi"}]
H = nx.Graph([(1, 2), (2, 3), (1, 3), (4, 5), (5, 6), (4, 6), (3, 4)])
parts_H = [{1, 2, 3}, {4, 5, 6}]

In [ ]:
def my_modularity(G, communities):
    m = G.number_of_edges()
    Q = 0.0
    for c in communities:
        l_c = None   # TODO
        d_c = None   # TODO
        Q += l_c / m - (d_c / (2 * m)) ** 2
    return Q


q_tri = None    # TODO
q_club = None   # TODO
q_one = None    # TODO

In [ ]:
# 核对（不用修改）
assert q_tri is not None and q_club is not None and q_one is not None, "q_tri、q_club、q_one 还没有赋值"
assert abs(q_tri - 0.357) < 0.001, "两个三角形的 Q 应约为 0.357 = 2×(3/7 − (7/14)²)；检查 m 是否用了全网总边数 7"
assert abs(q_club - nx.community.modularity(G, club)) < 1e-9, "q_club 与 nx.community.modularity 不一致；d_c 是社团内节点在全网中的度之和"
assert abs(q_club - 0.358) < 0.002, "空手道真实两派划分的 Q 应约为 0.358"
assert abs(q_one) < 1e-9, "整个网络作为一个社团时 Q = 1 − 1² = 0"
print("✓ 任务 1 通过")

## 任务 2：Girvan–Newman 算法

对空手道网络运行 Girvan–Newman（`nx.community.girvan_newman`），记录前 9 层划分（2 到 10 个社团）的社团数与模块度，存为列表 `levels`，每个元素是 `(社团数, Q)`。然后：

1. 找出 $Q$ 最大的一层：社团数 `best_n` 与模块度 `best_q`；
2. 第一次分裂（2 个社团）与真实两派相比，列出分错的节点 `wrong_gn`（列表）。

期望看到：$Q$ 在 5 个社团时最大（约 0.40）；第一次分裂只有个别节点与真实分派不同。

> **提示：** 生成器每次给出一层划分，到 10 个社团时 `break`；判断分错时，先找出 0 号教练所在的一侧。

In [ ]:
levels = []
first_split = None
for comms in nx.community.girvan_newman(G):
    comms = [set(c) for c in comms]
    pass   # TODO: 记录第一层到 first_split；把 (len(comms), Q) 追加到 levels；10 个社团时 break
best_n, best_q = None, None   # TODO
wrong_gn = None               # TODO

In [ ]:
# 核对（不用修改）
assert len(levels) == 9, "levels 应包含 2 到 10 个社团共 9 层；检查 break 条件"
assert [x[0] for x in levels] == list(range(2, 11)), "每一层社团数应依次为 2, 3, …, 10"
assert best_n == 5, "Q 最大的一层应有 5 个社团"
assert abs(best_q - 0.401) < 0.005, "最佳模块度应约为 0.40"
assert wrong_gn is not None and 1 <= len(wrong_gn) <= 3, "第一次分裂应只有个别节点（2 个左右）与真实分派不同；检查是否先确定了教练 0 所在的一侧"
print("✓ 任务 2 通过")

## 任务 3：谱方法：模块度矩阵的主特征向量

构造模块度矩阵 $B = A - \mathbf{k}\mathbf{k}^\top/2m$，求最大特征值对应的特征向量 `u`，按分量正负号把节点分成两组 `part_spec`，计算模块度 `q_spec`，并列出与真实两派不一致的节点 `wrong_spec`。

期望看到：只靠一次特征分解就几乎复现了俱乐部的分裂，只有 1 个节点分错；$Q\approx0.371$，略高于真实两派的 0.358。$|u_i|$ 最小的节点处在两派“中间”，归属最不确定。

> **提示：** `np.linalg.eigh` 返回的特征值从小到大排列，取最后一列 `vecs[:, -1]`。

In [ ]:
# 准备（直接运行）
nodes = sorted(G.nodes())
A = nx.to_numpy_array(G, nodelist=nodes)
k = A.sum(axis=1)
m = A.sum() / 2

In [ ]:
B = None           # TODO: A - np.outer(k, k) / (2 * m)
u = None           # TODO: 最大特征值的特征向量
part_spec = None   # TODO: [正号一组, 负号一组]（集合）
q_spec = None      # TODO
wrong_spec = None  # TODO

In [ ]:
# 核对（不用修改）
assert B is not None and u is not None, "B、u 还没有赋值"
assert np.allclose(np.asarray(B).sum(axis=1), 0), "模块度矩阵每行之和应为 0；检查分母是否是 2m"
u = np.asarray(u).ravel()
assert u.shape == (G.number_of_nodes(),), "u 应是长度为 34 的向量"
assert np.allclose(np.asarray(B) @ u, np.linalg.eigvalsh(B)[-1] * u, atol=1e-8), "u 应是最大特征值的特征向量；eigh 的特征值从小到大排列，要取最后一列"
assert part_spec is not None and sum(len(p) for p in part_spec) == 34, "part_spec 应覆盖全部 34 个节点"
assert abs(q_spec - 0.371) < 0.003, "谱二分的 Q 应约为 0.371"
assert wrong_spec is not None and len(wrong_spec) <= 2, "谱二分应只有 1 个节点与真实分派不同"
print("✓ 任务 3 通过")

## 任务 4：Louvain、分辨率参数与随机网络的模块度

**(a)** 用 Louvain（`nx.community.louvain_communities`，`seed=1`）在分辨率 $\gamma=0.5,1,1.5,2$ 下划分空手道网络，把社团数存入字典 `n_comm`（键为 $\gamma$）；记 $\gamma=1$ 时的模块度为 `Q_real`。

**(b)** 生成 20 个与空手道网络节点数、边数相同的 ER 网络（`nx.gnm_random_graph(34, 78, seed=s)`，s = 0–19），同样用 Louvain（`seed=s`）求 $Q$，存为数组 `Q_rand`；计算 $z = (Q_{\text{real}} - \overline{Q_{\text{rand}}})/\operatorname{sd}(Q_{\text{rand}})$，存为 `z_Q`。

期望看到：$\gamma$ 越大社团越多；随机网络的 $Q$ 平均约 0.35，与真实网络（约 0.42）相差并不大——$Q$ 的绝对值不能单独说明社团结构显著。
思考：更严格的零模型是保度随机网络（`nx.double_edge_swap`），结论会变吗？

In [ ]:
gammas = [0.5, 1.0, 1.5, 2.0]
n_comm = {}      # TODO
Q_real = None    # TODO
Q_rand = []      # TODO
z_Q = None       # TODO

In [ ]:
# 核对（不用修改）
assert set(n_comm) == set(gammas), "n_comm 应包含 4 个 γ 值"
assert n_comm[0.5] <= n_comm[1.0] <= n_comm[1.5] <= n_comm[2.0] and n_comm[2.0] > n_comm[0.5], "分辨率 γ 越大，社团应越多；检查是否把 resolution 参数传给了 louvain_communities"
assert Q_real is not None and 0.39 < Q_real < 0.43, "γ=1 时 Louvain 的 Q 应约为 0.42"
Q_rand = np.asarray(Q_rand, dtype=float)
assert Q_rand.shape == (20,), "Q_rand 应有 20 个值"
assert 0.28 < Q_rand.mean() < 0.40, "同规模 ER 网络的 Q 平均应约为 0.35——随机网络也能得到不低的模块度"
assert z_Q is not None and z_Q > 2, "真实网络的 Q 应明显高于随机网络（z 分数大于 2）"
print("✓ 任务 4 通过")

## 任务 5：比较两个划分：NMI 与 ARI

按专题页的定义，自己实现两个函数（不调用现成库）：

- `nmi(x, y)`：$\text{NMI} = 2I(X;Y)/[H(X)+H(Y)]$，以 2 为底的对数；
- `ari(x, y)`：调整兰德指数。

输入是两个等长的标签列表。先用专题页的手算例核对：6 个节点，真实划分 {0,1,2}{3,4,5}，检测划分 {0,1}{2,3,4,5}（节点 2 被放错），存为 `nmi_demo`、`ari_demo`，信息变差 $\text{VI}=H(X)+H(Y)-2I$ 存为 `vi_demo`。再比较空手道网络的 Louvain 划分（$\gamma=1$）与真实两派，存为 `nmi_club`、`ari_club`。

期望看到：手算例 NMI ≈ 0.479，ARI ≈ 0.324，VI = 1 比特——一个节点放错就让 ARI 降到 0.32。

> **提示：** 先用 `pd.crosstab` 或双重循环得到列联表 $n_{xy}$；$\binom{n}{2} = n(n-1)/2$。

In [ ]:
# 准备（直接运行）
import pandas as pd

x_demo = [0, 0, 0, 1, 1, 1]
y_demo = [0, 0, 1, 1, 1, 1]
louv = nx.community.louvain_communities(G, seed=1)
lab_louv = [next(i for i, c in enumerate(louv) if n in c) for n in nodes]
lab_club = [0 if n in club[0] else 1 for n in nodes]

In [ ]:
def entropy(counts):
    p = np.asarray(counts, dtype=float) / np.sum(counts)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))


def nmi(x, y):
    return None   # TODO


def ari(x, y):
    return None   # TODO


nmi_demo = ari_demo = vi_demo = None    # TODO
nmi_club = ari_club = None              # TODO

In [ ]:
# 核对（不用修改）
assert nmi(x_demo, y_demo) is not None and ari(x_demo, y_demo) is not None, "nmi、ari 函数还没有返回值"
assert abs(nmi_demo - 0.479) < 0.002, "手算例 NMI 应约为 0.479；检查互信息 I = H(X)+H(Y)−H(X,Y)"
assert abs(ari_demo - 0.324) < 0.002, "手算例 ARI 应约为 0.324；期望项为 Σ C(n_x,2)·Σ C(n_y,2) / C(n,2)"
assert abs(vi_demo - 1.0) < 1e-6, "手算例 VI 应为 1 比特；检查对数是否以 2 为底"
assert abs(nmi([0, 0, 1, 1], [1, 1, 0, 0]) - 1) < 1e-9 and abs(ari([0, 0, 1, 1], [5, 5, 7, 7]) - 1) < 1e-9, "只是标签名不同的两个相同划分，NMI 与 ARI 都应为 1"
assert 0 < nmi_club < 1 and -0.5 < ari_club < 1, "空手道 Louvain 划分与真实两派的 NMI 应在 0–1 之间"
print("✓ 任务 5 通过")

## 任务 6：小分析：节点角色——模块内度 z 与参与系数 P

以任务 5 的 Louvain 划分 `louv` 为准，按 Guimerà 与 Amaral（2005）的定义为每个节点计算：

- 模块内度 $z$ 分数 $z_i=(\kappa_i-\bar\kappa_{s_i})/\sigma_{\kappa_{s_i}}$，$\kappa_i$ 为连向本社团的边数（用本社团所有节点的均值与总体标准差）；
- 参与系数 $P_i=1-\sum_s (k_{is}/k_i)^2$。

结果存为两个字典 `z` 和 `P`（键为节点）。画 $z$–$P$ 散点图并标出几个突出的节点，用一两句话说明：哪些节点是“社团内枢纽”，哪些是“连接者”？套用 $z\ge2.5$ 的枢纽阈值会得出什么结论（专题页提醒：在小网络中直接套用阈值可能误导）？
本任务是开放的，只检查结构和取值范围；也可以换成任务 3 的谱二分划分比较。

In [ ]:
z = {}   # TODO
P = {}   # TODO

In [ ]:
# 核对（不用修改）
assert isinstance(z, dict) and isinstance(P, dict), "z 和 P 应为字典"
assert set(z) == set(G) and set(P) == set(G), "z 和 P 应包含全部 34 个节点"
zv = np.array([z[n] for n in nodes], dtype=float)
Pv = np.array([P[n] for n in nodes], dtype=float)
assert np.all(np.isfinite(zv)), "z 中有无穷或缺失值；社团内标准差为 0 时可设 z=0"
assert np.all((Pv >= -1e-9) & (Pv < 1)), "参与系数应在 [0, 1) 之间；检查分母是否为节点的总度 k_i"
assert abs(np.mean([z[n] for n in louv[0]])) < 1e-6, "同一社团内 z 分数的均值应为 0；检查是否用本社团的均值做中心化"
print("✓ 任务 6 通过")